# Simple RAG HotpotQA Evaluation

Evaluate [`simple_rag.py`](../simple_rag.py) (`SimpleVectorRAG`) on HotpotQA distractor dev retrieval.

**Metrics (k=10, aligned with `HotpotQA.ipynb`):**
- **Recall@10**: fraction of gold supporting **document titles** that appear among the top-10 retrieved chunks (micro-averaged over all gold docs).
- **MRR@10**: mean reciprocal rank of the first gold title in the top-10 retrieved chunk title list.

In [ ]:
from pathlib import Path
import os
import sys
import time

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")

In [ ]:
from simple_rag import SimpleVectorRAG, load_hotpot_documents

hotpot_file_candidates = [
    REPO_ROOT / "jupyter_notebooks" / "hotpot_dev_distractor_v1.json",
    REPO_ROOT / "hotpot_dev_distractor_v1.json",
]
HOTPOT_PATH = next((p for p in hotpot_file_candidates if p.exists()), hotpot_file_candidates[0])
print(f"Using HotpotQA file: {HOTPOT_PATH}")

NUM_SAMPLES = 500  # Set to None for the full distractor dev set.
TOP_K = 10
CHUNK_CHARS = 800
CHUNK_OVERLAP = 0
EMBEDDING_BACKEND = "local"  # Use "local" for e5-large-v2 or "api" for OpenAI embeddings.
EMBEDDING_MODEL = "text-embedding-3-small"
LOCAL_MODEL_PATH = REPO_ROOT.parent / "e5-large-v2"
LOCAL_DEVICE = "auto"
LOCAL_MAX_LENGTH = 512

def safe_path_part(value):
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in str(value))

DATASET_TAG = "all" if NUM_SAMPLES is None else str(NUM_SAMPLES)
MODEL_TAG = LOCAL_MODEL_PATH.name if EMBEDDING_BACKEND == "local" else EMBEDDING_MODEL
INDEX_TAG = "_".join([
    safe_path_part(DATASET_TAG),
    safe_path_part(EMBEDDING_BACKEND),
    safe_path_part(MODEL_TAG),
    f"chars{CHUNK_CHARS}",
    f"overlap{CHUNK_OVERLAP}",
])
INDEX_DIR = REPO_ROOT / "cache" / "simple_rag_hotpotqa" / INDEX_TAG
LOAD_INDEX_IF_EXISTS = True
FORCE_REBUILD_INDEX = False

documents, samples = load_hotpot_documents(HOTPOT_PATH, num_samples=NUM_SAMPLES)
print(f"Documents: {len(documents)}")
print(f"Samples: {len(samples)}")
print(f"Index directory: {INDEX_DIR}")

In [ ]:
rag = SimpleVectorRAG(
    index_dir=INDEX_DIR,
    embedding_backend=EMBEDDING_BACKEND,
    embedding_model=EMBEDDING_MODEL,
    local_model_path=LOCAL_MODEL_PATH,
    local_device=LOCAL_DEVICE,
    local_max_length=LOCAL_MAX_LENGTH,
    chunk_size=CHUNK_CHARS,
    chunk_overlap=CHUNK_OVERLAP,
    use_faiss_gpu=True,
)

index_ready = (INDEX_DIR / "index.faiss").exists() and (INDEX_DIR / "metadata.json").exists()

if index_ready and LOAD_INDEX_IF_EXISTS and not FORCE_REBUILD_INDEX:
    print("Loading saved FAISS index...")
    rag.load(INDEX_DIR)
    print(f"Loaded {len(rag._records)} chunk vectors from {INDEX_DIR}")
else:
    print(f"Building index ({EMBEDDING_BACKEND} embeddings; may take a while)...")
    t0 = time.time()
    n_chunks = rag.index_documents(documents, reset=True)
    rag.save(INDEX_DIR)
    print(f"Indexed {n_chunks} chunks in {time.time() - t0:.1f}s")
    print(f"Saved index to {INDEX_DIR}")

In [ ]:
def ranked_titles_from_records(records):
    return [rec.title for rec in records]


def gold_titles_for_sample(sample, documents):
    return [documents[doc_id]["title"] for doc_id in sample["gold_doc_ids"]]


def mrr_for_one_query_titles(ranked_titles, gold_titles, k=10):
    def norm(title):
        return " ".join(str(title).strip().lower().split())

    gold_set = {norm(title) for title in gold_titles if str(title).strip()}
    if not gold_set:
        return 0.0

    end = len(ranked_titles) if k is None else min(k, len(ranked_titles))
    for idx, title in enumerate(ranked_titles[:end], start=1):
        if norm(title) in gold_set:
            return 1.0 / idx
    return 0.0


def evaluate_hotpot_retrieval(rag, samples, documents, top_k=10):
    total_gold = 0
    total_hits = 0
    mrr_sum = 0.0
    n_queries = 0

    for sample in samples:
        gold_titles = gold_titles_for_sample(sample, documents)
        if not gold_titles:
            continue

        records = rag.query_sample_records(sample, top_k=top_k)
        ranked_titles = ranked_titles_from_records(records)

        ranked_norm = {t.strip().lower() for t in ranked_titles if t}
        for title in gold_titles:
            total_gold += 1
            if title.strip().lower() in ranked_norm:
                total_hits += 1

        mrr_sum += mrr_for_one_query_titles(ranked_titles, gold_titles, k=top_k)
        n_queries += 1

    recall_at_k = total_hits / total_gold if total_gold else 0.0
    mrr_at_k = mrr_sum / n_queries if n_queries else 0.0
    return {
        "recall@k": recall_at_k,
        "mrr@k": mrr_at_k,
        "total_gold_docs": total_gold,
        "total_hits": total_hits,
        "num_queries": n_queries,
    }

In [ ]:
t0 = time.time()
metrics = evaluate_hotpot_retrieval(rag, samples, documents, top_k=TOP_K)
elapsed = time.time() - t0

print(f"Evaluated {metrics['num_queries']} questions in {elapsed:.1f}s")
print(f"Gold supporting documents: {metrics['total_gold_docs']}")
print(f"Retrieved gold hits in top-{TOP_K}: {metrics['total_hits']}")
print(f"Recall@{TOP_K}: {metrics['recall@k']:.4f}")
print(f"MRR@{TOP_K}: {metrics['mrr@k']:.4f}")

In [ ]:
preview_idx = 0
sample = samples[preview_idx]
records = rag.query_sample_records(sample, top_k=TOP_K)

print("Question:", sample["question"])
print("Gold titles:", gold_titles_for_sample(sample, documents))
print(f"Top-{TOP_K} retrieved chunk titles:")
for rank, rec in enumerate(records, start=1):
    print(f"  {rank}. [{rec.doc_id}] {rec.title} (chunk {rec.chunk_idx})")